# 第14回 動向と評価：値とコードを一段ずつ読む

架空12事例を使い、正解数・回答数・全件数を分けて数えます。実サービスの性能を示す教材ではありません。

**開始：** 既存venvのカーネルを選び、初回は変更せず「カーネル再起動→すべて実行」。次に一節ずつ、予想してからShift+Enterで実行します。同梱Pythonのimportは不要で、外部通信も行いません。

**読み方：** 関数の定義は計算方法に名前を付けるセルです。続く短い実行セルが入力を渡します。表・図の表示コードは折り畳み指定です。表示は丸めますが計算には元の値を使います。

**完了の目印：** A7/12・B8/12。基準条件では方式A・しきい値80・費用32が適合候補。


In [ ]:
import sys
print('使用中のPython:',sys.executable)
print('版:',sys.version.split()[0])


In [ ]:
from copy import deepcopy


In [ ]:
import argparse


In [ ]:
import json


In [ ]:
DATA = {
    'A': {'normal': {'correct': [1,1,1,1,1,0], 'scores': [95,90,85,80,75,90]},
          'changed': {'correct': [1,1,0,0,0,0], 'scores': [95,80,75,65,55,45]}},
    'B': {'normal': {'correct': [1,1,1,1,0,0], 'scores': [90,85,80,75,70,60]},
          'changed': {'correct': [1,1,1,1,0,0], 'scores': [85,80,75,70,65,55]}},
}


In [ ]:
BASE = dict(threshold=0, normal_percent=50, review_cost=4, cost_a=1,
            cost_b=3, min_accuracy=80, min_coverage=50)


### 図表の準備（変更不要）

表示用の補助コードです。HTML/SVGを覚える必要はありません。折り畳み表示はNotebook環境によって異なります。


In [ ]:
from html import escape


In [ ]:
from IPython.display import HTML, display


In [ ]:
def learning_format(value):
    if isinstance(value, float):
        return f'{value:.6g}'
    if isinstance(value, (list, tuple)):
        return '['+', '.join(learning_format(item) for item in value)+']'
    if isinstance(value, dict):
        return '{'+', '.join(str(key)+': '+learning_format(item) for key,item in value.items())+'}'
    return str(value)


In [ ]:
def learning_table(title, columns, rows):
    header = ''.join('<th>'+escape(str(c))+'</th>' for c in columns)
    body = ''
    for row in rows:
        body += '<tr>'+''.join('<td>'+escape(learning_format(x))+'</td>' for x in row)+'</tr>'
    return '<div class="scroll"><table><caption>'+escape(title)+'</caption><tr>'+header+'</tr>'+body+'</table></div>'


In [ ]:
def learning_grid(title, matrix):
    rows = [(i,*row) for i,row in enumerate(matrix)]
    return learning_table(title,['行↓ 列→',*range(len(matrix[0]))],rows)


In [ ]:
def learning_plot(title, points):
    # 実計算の値だけを座標へ変換。点の間の値は補間して表示しない。
    xs, ys = [p[0] for p in points], [p[1] for p in points]
    xmin,xmax,ymin,ymax = min(xs),max(xs),min(0,min(ys)),max(ys)
    sx = lambda x: 100+(x-xmin)/max(1e-12,xmax-xmin)*500
    sy = lambda y: 240-(y-ymin)/max(1e-12,ymax-ymin)*190
    marks = ''
    for value in (ymin,(ymin+ymax)/2,ymax):
        marks += f'<path d="M100 {sy(value)} H600" stroke="#aaa"/><text x="0" y="{sy(value)+5}">{value:.4g}</text>'
    for x,y in points:
        marks += f'<circle cx="{sx(x)}" cy="{sy(y)}" r="4" fill="#155e75"><title>{x}: {y}</title></circle>'
    marks += f'<text x="100" y="275">{xmin}</text><text x="580" y="275">{xmax}</text>'
    return '<svg viewBox="0 0 700 310" role="img" aria-label="'+escape(title)+'"><text x="100" y="22">'+escape(title)+'</text>'+marks+'</svg>'


In [ ]:
def learning_show(report, reset=None):
    style = '<style>.learning{font:17px/1.65 sans-serif;color:#172b42;background:white;padding:12px}.learning .scroll{overflow-x:auto}.learning table{border-collapse:collapse;margin:14px 0}.learning td,.learning th{border:1px solid #8394a5;padding:6px 12px;white-space:normal}.learning caption{text-align:left;font-weight:bold}.learning svg{max-width:100%;height:auto}.learning .boards{display:flex;gap:20px;flex-wrap:wrap}.learning .result{border-left:4px solid #155e75;padding:10px}</style>'
    content = '<h3>'+escape(report['title'])+'</h3><p>入力：'+escape(learning_format(report['input']))+'</p>'
    content += learning_table('使う値 → 操作 → 結果',report['columns'],report['rows'])
    for title,columns,rows in report.get('tables',[]):
        content += '<details><summary>詳しい計算：'+escape(title)+'</summary>'+learning_table(title,columns,rows)+'</details>'
    content += '<div class="boards">'+''.join(learning_grid(title,values) for title,values in report.get('matrices',[]))+'</div>'
    for title,points in report.get('plots',[]):
        content += learning_plot(title,points)
    for label in ('result','comparison','explanation','reset'):
        name={'result':'結果','comparison':'一つ変えた比較','explanation':'読み取り方','reset':'戻す'}[label]
        content += '<p class="result">'+name+'：'+escape(reset if label=='reset' and reset is not None else learning_format(report[label]))+'</p>'
    display(HTML(style+'<div class="learning">'+content+'</div>'))


## 1. 事例と正解数

**今いる段階：** 事例→正誤→集計

**予想：** スコアと正解は同じ情報か

**操作：** example_methodを'A'から'B'へ変更し、同じセルを実行します。

**結果の目印：** Aは通常5/6、変化2/6、全体7/12

**戻す：** example_method='A'に戻して再実行します。

**説明する：** 分子と分母の対象をそろえる


### 計算の定義：`integer`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def integer(value, name):
    if type(value) is not int or not 0 <= value <= 100:
        raise ValueError(f'{name}: 0〜100の整数を指定してください')
    return value


### 計算の定義：`records`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def records(method, group='all'):
    if method not in ('A', 'B') or group not in ('all', 'normal', 'changed'):
        raise ValueError('未知の方式または条件です')
    rows = []
    for kind in ('normal', 'changed'):
        if group not in ('all', kind):
            continue
        for index, (correct, score) in enumerate(zip(DATA[method][kind]['correct'], DATA[method][kind]['scores']), 1):
            rows.append(dict(id=('N' if kind == 'normal' else 'C') + str(index),
                             group=kind, correct=correct, score=score))
    return rows


### 計算の定義：`aggregate`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def aggregate(method, threshold=0, group='all'):
    integer(threshold, 'threshold')
    rows = records(method, group)
    selected = [r for r in rows if r['score'] >= threshold]
    total, answered = len(rows), len(selected)
    correct = sum(r['correct'] for r in selected)
    return dict(total=total, correct=sum(r['correct'] for r in rows),
                answered=answered, correct_answered=correct, withheld=total-answered,
                accuracy=correct/answered if answered else None,
                coverage=answered/total, correct_per_total=correct/total,
                selected_ids=[r['id'] for r in selected])


### 計算の定義：`lesson_cases`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_cases(method='A'):
    rows = [(r['id'],r['group'],r['score'],r['correct']) for r in records(method)]
    tables = []
    for group in ('normal','changed','all'):
        a = aggregate(method,0,group)
        tables.append((group,['正解数','対象数','正解数÷対象数'],[(a['correct'],a['total'],a['accuracy'])]))
    return {'title':'12事例を一つずつ数える','question':'高いスコアなら必ず正解か',
            'input':f'架空方式{method}の固定12事例。実サービスの測定結果ではない。',
            'columns':['事例ID','対象条件','方式のスコア','正解なら1'],'rows':rows,'tables':tables,
            'result':aggregate(method),'comparison':'AのN6はスコア90でも不正解。Bへ変更して各条件と全体を比較する。',
            'explanation':'N1〜N6は通常条件、C1〜C6は変化条件。スコアと正解ラベルは別。正解率の分母は対象事例数。',
            'reset':'method="A"で再実行。固定データを変更しない。'}


In [ ]:
example_method = 'A'
program_observation_cases = lesson_cases(example_method)
learning_show(program_observation_cases, reset="example_method='A'に戻して再実行します。")


## 2. 対象集団の比率

**今いる段階：** 群別正解率→重み→合計

**予想：** 対象割合で順位は変わるか

**操作：** example_normal_percentを50から75へ変更し、同じセルを実行します。

**結果の目印：** Aは通常割合75%で17/24

**戻す：** example_normal_percent=50に戻して再実行します。

**説明する：** 実測と加重比較を分ける


### 計算の定義：`configuration`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def configuration(config=None):
    if config is None:
        return deepcopy(BASE)
    if not isinstance(config, dict) or set(config) - set(BASE):
        raise ValueError('未知の設定です')
    result = dict(BASE, **config)
    for key, value in result.items():
        integer(value, key)
    return result


### 計算の定義：`choose`

固定候補の条件照合。丸め前の整数積で境界を比較する。


In [ ]:
def choose(candidates, min_accuracy, min_coverage):
    """固定候補の条件照合。丸め前の整数積で境界を比較する。"""
    integer(min_accuracy, 'min_accuracy')
    integer(min_coverage, 'min_coverage')
    eligible = [deepcopy(c) for c in candidates if c['answered'] > 0
                and c['correct_answered']*100 >= min_accuracy*c['answered']
                and c['answered']*100 >= min_coverage*c['total']]
    eligible.sort(key=lambda c: (c['cost'], -c['coverage'], -c['accuracy'], c['method'], c['threshold']))
    return eligible[0] if eligible else None


### 計算の定義：`evaluate`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def evaluate(config=None):
    c = configuration(config)
    groups, mixture, answers, candidates = {}, {}, {}, []
    for method in ('A', 'B'):
        groups[method] = {kind: aggregate(method, 0, kind) for kind in ('all', 'normal', 'changed')}
        normal, changed = groups[method]['normal'], groups[method]['changed']
        mixture[method] = (normal['correct']*c['normal_percent'] + changed['correct']*(100-c['normal_percent']))/600
        cost = c['cost_a' if method == 'A' else 'cost_b']
        answers[method] = aggregate(method, c['threshold'])
        answers[method]['cost'] = 12*cost + answers[method]['withheld']*c['review_cost']
        for threshold in (0, 80, 100):
            row = dict(method=method, threshold=threshold, **aggregate(method, threshold))
            row['cost'] = 12*cost + row['withheld']*c['review_cost']
            row['eligible'] = (row['answered'] > 0 and row['correct_answered']*100 >= c['min_accuracy']*row['answered']
                               and row['answered']*100 >= c['min_coverage']*12)
            candidates.append(row)
    return dict(config=c, groups=groups, mixture=mixture, answers=answers,
                candidates=candidates, best=choose(candidates, c['min_accuracy'], c['min_coverage']))


### 計算の定義：`lesson_mixture`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_mixture(normal_percent=50):
    result = evaluate({'normal_percent':normal_percent})
    rows = []
    for method in ('A','B'):
        n,c = result['groups'][method]['normal'],result['groups'][method]['changed']
        rows.append((method,n['correct'],c['correct'],normal_percent/100,1-normal_percent/100,n['accuracy']*normal_percent/100,c['accuracy']*(100-normal_percent)/100,result['mixture'][method]))
    return {'title':'対象集団の割合を変えて比べる','question':'通常条件が多い集団でも順位は同じか',
            'input':f'通常{normal_percent}%・変化{100-normal_percent}%。各群の6事例はそのまま。',
            'columns':['方式','通常の正解/6','変化の正解/6','通常の重み','変化の重み','通常の寄与','変化の寄与','加重正解率'],'rows':rows,
            'result':result['mixture'],'comparison':'通常割合50→75でAの加重正解率は7/12→17/24。Bは両群4/6なので変わらない。',
            'explanation':'新しい実測事例を作ったのではなく、固定した群別正解率を割合で重み付けした比較。',
            'reset':'normal_percent=50で再実行。'}


In [ ]:
example_normal_percent = 50
program_observation_mixture = lesson_mixture(example_normal_percent)
learning_show(program_observation_mixture, reset='example_normal_percent=50に戻して再実行します。')


## 3. 回答と保留

**今いる段階：** しきい値→対象選択→三つの分母

**予想：** 回答0の正解率は何か

**操作：** example_thresholdを80から100へ変更し、同じセルを実行します。

**結果の目印：** 回答0ではNone、回答率は0

**戻す：** example_threshold=80に戻して再実行します。

**説明する：** 回答内正解率だけでは不十分な理由


### 計算の定義：`lesson_abstention`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_abstention(threshold=80):
    a = aggregate('A',threshold)
    rows = [(r['id'],r['score'],r['correct'],'回答' if r['score']>=threshold else '保留') for r in records('A')]
    return {'title':'保留すると、三つの分母がどう変わるか','question':'回答内正解率が高ければ全事例を解決したと言えるか',
            'input':f'方式A、スコア≥{threshold}だけ回答。',
            'columns':['事例','スコア','正解','判定'],'rows':rows,
            'tables':[('集計の分子と分母',['指標','分子','分母','値'],[('回答内正解率',a['correct_answered'],a['answered'],a['accuracy']),('回答率',a['answered'],a['total'],a['coverage']),('全事例あたり正解',a['correct_answered'],a['total'],a['correct_per_total'])])],
            'result':a,'comparison':'しきい値80では6/7正解・7/12回答。100では回答0、回答内正解率はNone（定義できない）。',
            'explanation':'保留を正解にも誤答にも加えず、回答率と併記する。回答0の正解率を0%や100%と補わない。',
            'reset':'threshold=80で再実行。'}


In [ ]:
example_threshold = 80
program_observation_abstention = lesson_abstention(example_threshold)
learning_show(program_observation_abstention, reset='example_threshold=80に戻して再実行します。')


## 4. 費用の内訳

**今いる段階：** 件数×単価→合計

**予想：** 何件に処理費用がかかるか

**操作：** example_review_costを4から10へ変更し、同じセルを実行します。

**結果の目印：** Aは12×1＋5×4=32

**戻す：** example_review_cost=4に戻して再実行します。

**説明する：** 保留にも処理費用がかかる理由


### 計算の定義：`lesson_cost`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_cost(review_cost=4):
    result = evaluate({'threshold':80,'review_cost':review_cost})
    rows = []
    for method,unit in [('A',BASE['cost_a']),('B',BASE['cost_b'])]:
        a=result['answers'][method]
        rows.append((method,12,unit,12*unit,a['withheld'],review_cost,a['withheld']*review_cost,a['cost']))
    return {'title':'全件の処理費用と保留の確認費用を足す','question':'安い方式でも人の確認が多いとどうなるか',
            'input':f'しきい値80、人の確認1件={review_cost}費用単位。架空の費用。',
            'columns':['方式','全件数','処理単価','処理費用','保留数','確認単価','確認費用','総費用'],'rows':rows,
            'result':result['answers'],'comparison':'確認単価4→10でAの総費用32→62、Bは64→106。',
            'explanation':'保留事例も先に方式を通しているため、方式の処理費用は回答数ではなく全12件にかかる。',
            'reset':'review_cost=4で再実行。'}


In [ ]:
example_review_cost = 4
program_observation_cost = lesson_cost(example_review_cost)
learning_show(program_observation_cost, reset='example_review_cost=4に戻して再実行します。')


## 5. 条件を満たす候補の選択

**今いる段階：** 候補→適合判定→費用比較

**予想：** 適合候補がなければどう報告するか

**操作：** example_coverageを50から100へ変更し、同じセルを実行します。

**結果の目印：** 回答率100%以上では適合候補なし

**戻す：** example_coverage=50に戻して再実行します。

**説明する：** 条件確認と順位付けを分ける


### 計算の定義：`lesson_choice`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_choice(min_coverage=50):
    result = evaluate({'min_coverage':min_coverage})
    rows = []
    for c in result['candidates']:
        rows.append((c['method'],c['threshold'],c['correct_answered'],c['answered'],c['accuracy'],c['coverage'],c['cost'],c['eligible']))
    return {'title':'条件に合う候補だけで費用を比べる','question':'最安だけを見て決めてよいか',
            'input':f'回答内正解率80%以上、回答率{min_coverage}%以上。候補はA/B×しきい値0/80/100。',
            'columns':['方式','しきい値','回答中正解','回答数/12','回答内正解率','回答率','費用','適合'],'rows':rows,
            'result':result['best'] if result['best'] else '適合候補なし',
            'comparison':'回答率50→100では適合候補がなくなる。条件を黙って緩めない。',
            'explanation':'丸め前の整数積で条件判定し、適合候補の中で費用を比較する。架空12事例の範囲での選択であり、実製品の性能順位ではない。',
            'reset':'min_coverage=50で再実行。'}


In [ ]:
example_coverage = 50
program_observation_choice = lesson_choice(example_coverage)
learning_show(program_observation_choice, reset='example_coverage=50に戻して再実行します。')


## 基準結果と既存の変更実験

ここまでの図表を、従来の基準出力と照合します。以下の実験も入力のコピーを使います。定義を変更した場合は再起動して全セル実行してください。


### 計算の定義：`inspect_case`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def inspect_case(method, case_id):
    for row in records(method):
        if row['id'] == case_id:
            return row
    raise ValueError('未知のケースです')


### 計算の定義：`experiment_configs`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def experiment_configs():
    cases = {}
    for name, field, value, base_threshold in (
        ('threshold', 'threshold', 80, 0), ('mixture', 'normal_percent', 75, 0),
        ('review', 'review_cost', 10, 80), ('cost_b', 'cost_b', 1, 80),
        ('coverage', 'min_coverage', 40, 0), ('accuracy', 'min_accuracy', 90, 0)):
        base = dict(BASE, threshold=base_threshold)
        changed = dict(base, **{field: value})
        cases[name] = dict(base=base, changed=changed)
    return cases


### 計算の定義：`rounded`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def rounded(value):
    if isinstance(value, dict):
        return {k: rounded(v) for k, v in value.items()}
    if isinstance(value, list):
        return [rounded(v) for v in value]
    return round(value, 6) if isinstance(value, float) else value


### 計算の定義：`demo`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def demo():
    return rounded(dict(baseline=evaluate(), changes={name: dict(before=evaluate(case['base']), after=evaluate(case['changed']))
                                                    for name, case in experiment_configs().items()}))


## 基準結果
A 7/12・B 8/12。候補選択はA・しきい値80・費用32。表示値だけで安全性は判断できません。


In [ ]:
baseline_config = deepcopy(BASE)
baseline_data = deepcopy(DATA)
baseline = evaluate(baseline_config)
baseline_result = demo()
print(json.dumps(rounded(baseline), ensure_ascii=False, indent=2))


## 操作テーマ1：ケースを確認

問い・予想：操作教材と同じ結果になるでしょうか。実行前に予想を書きます。

手順：次のセルを実行。想定結果：N6はスコア90で不正解。

説明：変えた条件・分母・限界を挙げてください。復帰：再実行で同じ結果に戻ります。これは基準入力を編集しません。


In [ ]:
observation_case = inspect_case('A', 'N6')
print(observation_case)


## 操作テーマ2：条件別に集計

問い・予想：操作教材と同じ結果になるでしょうか。実行前に予想を書きます。

手順：次のセルを実行。想定結果：通常条件A5/6・B4/6。

説明：変えた条件・分母・限界を挙げてください。復帰：再実行で同じ結果に戻ります。これは基準入力を編集しません。


In [ ]:
observation_groups = {m: aggregate(m, 0, 'normal') for m in ('A', 'B')}
print(observation_groups)


## 操作テーマ3：比率を仮定

問い・予想：操作教材と同じ結果になるでしょうか。実行前に予想を書きます。

手順：次のセルを実行。想定結果：推定A17/24・B2/3。

説明：変えた条件・分母・限界を挙げてください。復帰：再実行で同じ結果に戻ります。これは基準入力を編集しません。


In [ ]:
observation_mixture = evaluate({'normal_percent': 75})
print(observation_mixture['mixture'])


## 操作テーマ4：回答と保留

問い・予想：操作教材と同じ結果になるでしょうか。実行前に予想を書きます。

手順：次のセルを実行。想定結果：A6/7・回答率7/12、B5/5・回答率5/12。

説明：変えた条件・分母・限界を挙げてください。復帰：再実行で同じ結果に戻ります。これは基準入力を編集しません。


In [ ]:
observation_abstain = evaluate({'threshold': 80})
print(observation_abstain['answers'])


## 操作テーマ5：確認費用を計算

問い・予想：操作教材と同じ結果になるでしょうか。実行前に予想を書きます。

手順：次のセルを実行。想定結果：しきい値80の比較基準から確認費用だけ変更し、62・106。

説明：変えた条件・分母・限界を挙げてください。復帰：再実行で同じ結果に戻ります。これは基準入力を編集しません。


In [ ]:
observation_costs = evaluate({'threshold': 80, 'review_cost': 10})
print({m: observation_costs['answers'][m]['cost'] for m in ('A', 'B')})


## 操作テーマ6：要件を照合

問い・予想：操作教材と同じ結果になるでしょうか。実行前に予想を書きます。

手順：次のセルを実行。想定結果：必要回答率50%も満たす候補はなくNone。

説明：変えた条件・分母・限界を挙げてください。復帰：再実行で同じ結果に戻ります。これは基準入力を編集しません。


In [ ]:
observation_choice = evaluate({'min_accuracy': 90})
print(observation_choice['best'])


## 変更1：しきい値0→80

問い・予想：何が変わり、何が変わらないか、実行前に手元へ書きましょう。

手順：この例の基準設定をコピーして一項目だけ変更します。想定結果：Aは6/7・回答率7/12、Bは5/5・回答率5/12。

説明：分母・変えた条件・測っていない事項を挙げてください。復帰：このセルを再実行。全体はカーネル再起動後の全セル実行。


In [ ]:
trial_threshold_base = deepcopy(experiment_configs()['threshold']['base'])
trial_threshold_config = deepcopy(trial_threshold_base)
trial_threshold_config.update({k: v for k, v in experiment_configs()['threshold']['changed'].items() if v != trial_threshold_base[k]})
trial_threshold_before = evaluate(trial_threshold_base)
trial_threshold = evaluate(trial_threshold_config)
print(json.dumps(rounded(trial_threshold), ensure_ascii=False, indent=2))


## 変更2：通常条件50→75%

問い・予想：何が変わり、何が変わらないか、実行前に手元へ書きましょう。

手順：この例の基準設定をコピーして一項目だけ変更します。想定結果：推定A17/24・B2/3。元の固定結果は変わらない。

説明：分母・変えた条件・測っていない事項を挙げてください。復帰：このセルを再実行。全体はカーネル再起動後の全セル実行。


In [ ]:
trial_mixture_base = deepcopy(experiment_configs()['mixture']['base'])
trial_mixture_config = deepcopy(trial_mixture_base)
trial_mixture_config.update({k: v for k, v in experiment_configs()['mixture']['changed'].items() if v != trial_mixture_base[k]})
trial_mixture_before = evaluate(trial_mixture_base)
trial_mixture = evaluate(trial_mixture_config)
print(json.dumps(rounded(trial_mixture), ensure_ascii=False, indent=2))


## 変更3：確認費用4→10（基準しきい値80）

問い・予想：何が変わり、何が変わらないか、実行前に手元へ書きましょう。

手順：この例の基準設定をコピーして一項目だけ変更します。想定結果：総費用A62・B106。

説明：分母・変えた条件・測っていない事項を挙げてください。復帰：このセルを再実行。全体はカーネル再起動後の全セル実行。


In [ ]:
trial_review_base = deepcopy(experiment_configs()['review']['base'])
trial_review_config = deepcopy(trial_review_base)
trial_review_config.update({k: v for k, v in experiment_configs()['review']['changed'].items() if v != trial_review_base[k]})
trial_review_before = evaluate(trial_review_base)
trial_review = evaluate(trial_review_config)
print(json.dumps(rounded(trial_review), ensure_ascii=False, indent=2))


## 変更4：B単価3→1（基準しきい値80）

問い・予想：何が変わり、何が変わらないか、実行前に手元へ書きましょう。

手順：この例の基準設定をコピーして一項目だけ変更します。想定結果：B総費用40。A32は変わらない。

説明：分母・変えた条件・測っていない事項を挙げてください。復帰：このセルを再実行。全体はカーネル再起動後の全セル実行。


In [ ]:
trial_cost_b_base = deepcopy(experiment_configs()['cost_b']['base'])
trial_cost_b_config = deepcopy(trial_cost_b_base)
trial_cost_b_config.update({k: v for k, v in experiment_configs()['cost_b']['changed'].items() if v != trial_cost_b_base[k]})
trial_cost_b_before = evaluate(trial_cost_b_base)
trial_cost_b = evaluate(trial_cost_b_config)
print(json.dumps(rounded(trial_cost_b), ensure_ascii=False, indent=2))


## 変更5：必要回答率50→40%

問い・予想：何が変わり、何が変わらないか、実行前に手元へ書きましょう。

手順：この例の基準設定をコピーして一項目だけ変更します。想定結果：適合候補はA80・B80。選択はA80のまま。

説明：分母・変えた条件・測っていない事項を挙げてください。復帰：このセルを再実行。全体はカーネル再起動後の全セル実行。


In [ ]:
trial_coverage_base = deepcopy(experiment_configs()['coverage']['base'])
trial_coverage_config = deepcopy(trial_coverage_base)
trial_coverage_config.update({k: v for k, v in experiment_configs()['coverage']['changed'].items() if v != trial_coverage_base[k]})
trial_coverage_before = evaluate(trial_coverage_base)
trial_coverage = evaluate(trial_coverage_config)
print(json.dumps(rounded(trial_coverage), ensure_ascii=False, indent=2))


## 変更6：必要正答率80→90%

問い・予想：何が変わり、何が変わらないか、実行前に手元へ書きましょう。

手順：この例の基準設定をコピーして一項目だけ変更します。想定結果：必要回答率50%も満たす候補はない。

説明：分母・変えた条件・測っていない事項を挙げてください。復帰：このセルを再実行。全体はカーネル再起動後の全セル実行。


In [ ]:
trial_accuracy_base = deepcopy(experiment_configs()['accuracy']['base'])
trial_accuracy_config = deepcopy(trial_accuracy_base)
trial_accuracy_config.update({k: v for k, v in experiment_configs()['accuracy']['changed'].items() if v != trial_accuracy_base[k]})
trial_accuracy_before = evaluate(trial_accuracy_base)
trial_accuracy = evaluate(trial_accuracy_config)
print(json.dumps(rounded(trial_accuracy), ensure_ascii=False, indent=2))


## 基準の保持とエラー対処

空欄や文字列ではなく0〜100の整数を指定します。`ValueError`は値・キーを見直します。`NameError`は上の定義セルが未実行です。カーネルを選べない場合は同梱guide.htmlの手順を確認します。全件保留のNoneは正常です。

問い：同じ固定データを再集計したことは、新しい独立評価を行ったことになるでしょうか。


In [ ]:
assert baseline_config == BASE
assert baseline_data == DATA
assert baseline == evaluate(BASE)
assert baseline_result == demo()
print('基準入力・結果を保持しました')


## 振り返りと復帰

入力・行った処理・結果・変更前との差を、自分の言葉で説明してください。観察セルは毎回基準のコピーから計算するため、同じセルを再実行しても前の実験の続きにはなりません。NameErrorや実行順が不明な場合はカーネル再起動→全セル実行。元の配布ファイルを残し、変更版は別名保存します。

Python版は`python ai_evaluation.py --explain all`で同じ計算表を表示します。各テーマは`--explain cases`のように指定できます。従来のコマンドと固定期待値も維持しています。
